###### Widgets

In [0]:
dbutils.widgets.dropdown("environment", "dev", ["dev", "azure"])
dbutils.widgets.text("source_name", "orders")
dbutils.widgets.text("storage_account", "dlspl21databricks")
dbutils.widgets.text("storage_container", "yanquiel")
dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("bronze_schema", "yanquiel_bronze")
dbutils.widgets.dropdown("file_format", "csv", ["csv", "json", "parquet"])

###### Read Widget Values

In [0]:
environment = dbutils.widgets.get("environment")
source_name = dbutils.widgets.get("source_name")
storage_account = dbutils.widgets.get("storage_account")
storage_container = dbutils.widgets.get("storage_container")
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
file_format = dbutils.widgets.get("file_format")
raw_container = dbutils.widgets.get("raw_container")

##### Resolve Paths by Environment

In [0]:
if environment == "dev":
    source_path = f"/Volumes/{catalog}/default/raw_data/{source_name}"
    checkpoint_path = f"/Volumes/{catalog}/default/raw_data/_checkpoints/{source_name}"
elif environment == "azure":
    source_path = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net/{raw_container}/{source_name}"
    checkpoint_path = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net/{raw_container}/_checkpoints/{source_name}"

bronze_table = f"{catalog}.{bronze_schema}.{source_name}"

print("Environment:", environment)
print("Source:", source_path)
print("Checkpoint:", checkpoint_path)
print("Table:", bronze_table)

##### Read Source Files (Auto Loader)

In [0]:
from pyspark.sql.functions import current_timestamp, current_date, col

df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", file_format)
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
    .option("header", "true")
    .load(source_path)
)

##### Add Metadata Columns

In [0]:
df_bronze = (
    df_raw
    .select(
        "*",
        col("_metadata.file_name").alias("source_filename")
    )
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

##### Write to Bronze

In [0]:
query = (
    df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

##### Verify Result

In [0]:
display(spark.table(bronze_table))

In [0]:
print(spark.conf.get("spark.databricks.clusterUsageTags.clusterId"))

In [0]:
spark.sql("SELECT COUNT(*), MIN(ingestion_timestamp), MAX(ingestion_timestamp) FROM dbr_dev.yanquiel_bronze.orders").show()